# Brain MRI Classification and Tumour Localisation
### Image Processing + Deep Learning (VGG16 vs ResNet18)

In [ ]:
pip install opencv-python

In [ ]:
from pathlib import Path

# ---- EDIT ONLY THESE TWO PATHS ----
DATA_DIR = Path("demo_images")    # folder that contains Training and Testing
VGG_PATH = Path("ml/weights/vgg16.pth")           # the VGG16 model you trained earlier

OUT_DIR = Path("ipcv_outputs")     # figures and results are saved here (created automatically)
SAVE_RESNET_WEIGHTS = False        # set True to also save the trained ResNet18 inside OUT_DIR

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

IMG_SIZE        = 224
BATCH_SIZE      = 16
VAL_SPLIT       = 0.15     # 15% of Training is kept for validation
SEED            = 42
FREEZE_EPOCHS   = 3        # phase 1: train only the last layer
FINETUNE_EPOCHS = 6        # phase 2: train the whole network slowly
LR_HEAD         = 1e-3
LR_FINETUNE     = 2e-5
PATIENCE        = 4        # stop early if validation does not improve for 4 epochs

# ImageNet mean and std (the pre-trained models expect this)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [ ]:
import copy, json, random, time
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import imagehash

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# fix the random seeds so that results can be repeated
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (device.type == "cuda")      # mixed precision makes GPU training faster

OUT_DIR.mkdir(exist_ok=True)

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Step 1: Load the dataset and look at the images
*Syllabus module 1: image acquisition, loading, displaying.*

The dataset has two folders, `Training` and `Testing`. Each has four class folders. First we make a list of (image path, class name) for every image.

In [ ]:
def get_image_list(folder):
    # returns a list of (image_path, class_name)
    image_list = []
    for class_name in CLASS_NAMES:
        class_folder = folder / class_name
        for file in sorted(class_folder.iterdir()):
            if file.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                image_list.append((file, class_name))
    return image_list

train_items = get_image_list(DATA_DIR / "Training")
test_items  = get_image_list(DATA_DIR / "Testing")

print("Training images:", len(train_items))
print("Testing images :", len(test_items))

# count images per class
train_counts = [sum(1 for _, c in train_items if c == name) for name in CLASS_NAMES]
test_counts  = [sum(1 for _, c in test_items  if c == name) for name in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
plt.figure(figsize=(7, 3.5))
plt.bar(x - 0.2, train_counts, width=0.4, label="Training")
plt.bar(x + 0.2, test_counts,  width=0.4, label="Testing")
plt.xticks(x, CLASS_NAMES)
plt.ylabel("Number of images")
plt.title("Images per class")
plt.legend()
plt.show()

In [ ]:
# show one image of each class and print its properties
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, name in zip(axes, CLASS_NAMES):
    path = [p for p, c in train_items if c == name][0]
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)     # read the image exactly as stored
    print(name, "| shape:", img.shape, "| type:", img.dtype, "| min:", img.min(), "| max:", img.max())
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)         # OpenCV reads colour as BGR
    ax.imshow(img, cmap="gray")
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
sample = random.Random(SEED).sample(train_items, 300)

size_counter = Counter()
colour_images = 0
for path, _ in sample:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    size_counter[img.shape[:2]] += 1            # (height, width)
    if len(img.shape) == 3:
        colour_images += 1

print("Different image sizes found:", len(size_counter))
for size, count in size_counter.most_common(5):
    print("  size", size, "->", count, "images")
print("Images stored with 3 channels:", colour_images, "out of 300")

In [ ]:

plt.figure(figsize=(8, 4))
for name in CLASS_NAMES:
    path = [p for p, c in train_items if c == name][0]
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    plt.plot(hist, label=name)
plt.xlabel("Pixel intensity")
plt.ylabel("Number of pixels")
plt.title("Intensity histograms")
plt.legend()
plt.show()

## Step 3: Preprocessing decisions
*Syllabus modules 2, 3 and 4.*

We only keep the steps we can justify.

| Step | Applied? | Reason |
|---|---|---|
| Convert to 3 channels | Yes | Pre-trained models expect 3 channels. MRI is gray, so the same channel is repeated. |
| Resize to 224 x 224 | Yes | Image sizes differ. 224 is the size VGG16 and ResNet18 were pre-trained on. |
| Scale to 0-1, then normalise with ImageNet mean and std | Yes | The pre-trained weights expect this intensity range. It also makes training stable. |
| Random flip, small rotation, brightness and contrast change | Yes, training only | Gives more variety and reduces overfitting. Validation and test images are never changed. |
| Remove duplicate images | Yes | Duplicates between train and test would make the score look better than it is. |
| Histogram equalisation (CLAHE) | No (tested below) | Can boost noise and changes the intensity statistics the pre-trained models expect. |
| Median or Gaussian filter | No (tested below) | The first CNN layers already learn smoothing and edge filters. |
| Otsu threshold, morphology, Canny edges on the raw MRI | No (tested below) | Not needed for classification. We use Otsu and morphology in Step 10, on the Grad-CAM heatmap instead. |

Another reason to keep the preprocessing simple: **VGG16 was trained without the extra steps**. If we added them only for ResNet18, the comparison would not be fair.

In [ ]:

path = train_items[0][0]

step0 = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)                        # read as gray
step1 = cv2.resize(step0, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)   # resize
step2 = cv2.cvtColor(step1, cv2.COLOR_GRAY2RGB)                            # 1 channel -> 3 channels
step3 = step2.astype(np.float32) / 255.0                                   # scale to 0-1
step4 = (step3 - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)   # normalise

print("original   :", step0.shape, step0.dtype, "range", step0.min(), "-", step0.max())
print("resized    :", step1.shape)
print("3 channels :", step2.shape)
print("scaled 0-1 :", step3.shape, "range", round(float(step3.min()), 2), "-", round(float(step3.max()), 2))
print("normalised :", step4.shape, "range", round(float(step4.min()), 2), "-", round(float(step4.max()), 2))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(step0, cmap="gray"); axes[0].set_title("Original " + str(step0.shape))
axes[1].imshow(step2);              axes[1].set_title("Resized, 3 channels")
axes[2].imshow(np.clip(step4, 0, 1)); axes[2].set_title("After normalisation (display only)")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
# Steps we TESTED but did NOT apply (to show we considered them)
gray = step1                                                    # 224 x 224 gray image

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(gray)                                   # histogram equalisation
img_median = cv2.medianBlur(gray, 5)                            # noise removal
_, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)      # Otsu threshold
otsu_clean = cv2.morphologyEx(otsu, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))  # remove small noise
edges = cv2.Canny(gray, 100, 200)                               # edges

images = [gray, img_clahe, img_median, otsu_clean, edges]
titles = ["Original", "CLAHE", "Median filter", "Otsu + opening", "Canny edges"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, im, t in zip(axes, images, titles):
    ax.imshow(im, cmap="gray")
    ax.set_title(t)
    ax.axis("off")
plt.show()

## Step 4: Remove duplicate images
The same MRI slice can appear twice in the dataset, even once in Training and once in Testing. If a test image was also seen during training, the test score is not honest. 

We give every image a short **fingerprint (perceptual hash)**. Images with the same fingerprint are duplicates, and we remove the extra copies from the training data. This step only compares fingerprints. It does not change the Testing set.

In [ ]:
def get_hash(path):
    img = Image.open(path).convert("L")
    return str(imagehash.phash(img))

# fingerprints of all training images
train_hashes = {}
seen_hashes = {}
duplicate_paths = []              # extra copies inside Training
for path, label in tqdm(train_items, desc="Training hashes"):
    h = get_hash(path)
    train_hashes[path] = h
    if h in seen_hashes:
        duplicate_paths.append(path)
    else:
        seen_hashes[h] = path

# fingerprints of all test images
test_hash_set = set()
for path, label in tqdm(test_items, desc="Testing hashes"):
    test_hash_set.add(get_hash(path))

# training images that also exist in the test set
leak_paths = [p for p, _ in train_items if train_hashes[p] in test_hash_set]

remove_set = set(duplicate_paths) | set(leak_paths)
clean_items = [(p, c) for p, c in train_items if p not in remove_set]

print("Duplicates inside Training       :", len(duplicate_paths))
print("Training images also in Testing  :", len(leak_paths))
print("Images removed from training     :", len(remove_set))
print("Clean training images left       :", len(clean_items))

## Step 5: Split the data and build the data loaders
*Syllabus module 2: brightness/contrast change as intensity transformation.*

* **Training (85%)**: the model learns from these images.
* **Validation (15%)**: used during training to pick the best epoch and to calibrate the confidence.
* **Testing**: untouched until Step 9.

The split uses the same seed and the same cleaning code that was used for VGG16, so **both models should use the same training and validation images**. Step 6 checks this using the accuracy saved with VGG16.

In [ ]:
labels_list = [c for _, c in clean_items]
train_list, val_list = train_test_split(clean_items, test_size=VAL_SPLIT,
                                        stratify=labels_list, random_state=SEED)
print("Train:", len(train_list), "| Validation:", len(val_list))

In [ ]:
# augmentation: random changes, used only for training images
augment = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

train_tf = transforms.Compose([augment, transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf  = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# preview: the same image augmented 5 times
pil_img = Image.open(train_list[0][0]).convert("RGB")
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE))); axes[0].set_title("Original")
for i in range(1, 6):
    axes[i].imshow(augment(pil_img)); axes[i].set_title("Augmented " + str(i))
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

class MRIDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items              # list of (path, class_name)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, class_name = self.items[i]
        img = Image.open(path).convert("RGB")       # gray MRI -> 3 channels
        return self.transform(img), CLASS_TO_IDX[class_name]

train_loader = DataLoader(MRIDataset(train_list, train_tf), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)
val_loader   = DataLoader(MRIDataset(val_list, eval_tf), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
print("Batches per epoch:", len(train_loader))

In [ ]:
# helper functions used for both models

@torch.no_grad()
def get_logits(model, loader):
    # runs the model on a data loader and returns raw scores (logits) and true labels
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        outputs = model(images.to(device))
        all_logits.append(outputs.float().cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

def find_temperature(logits, labels):
    # try many values of T and keep the one with the lowest loss on the validation data
    best_T, best_loss = 1.0, 1e9
    for T in np.arange(0.5, 5.01, 0.05):
        loss = nn.functional.cross_entropy(logits / float(T), labels).item()
        if loss < best_loss:
            best_T, best_loss = float(T), loss
    return best_T

def calibration_error(probs, labels, n_bins=15):
    # Expected Calibration Error: how far the confidence is from the real accuracy
    confidence = probs.max(1)
    correct = (probs.argmax(1) == labels)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.any():
            ece += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())
    return float(ece)

## Step 6: Load the saved VGG16 model
*Syllabus module 5: pre-trained CNN models.*

VGG16 was already trained in the previous notebook, so we only load it. We rebuild the same network (VGG16 with a 4-class last layer) and load the saved weights.

In [ ]:
vgg_ckpt = torch.load(VGG_PATH, map_location="cpu")

# the file is either a dictionary with extra information, or only the weights
if isinstance(vgg_ckpt, dict) and "state_dict" in vgg_ckpt:
    vgg_weights = vgg_ckpt["state_dict"]
    VGG_TEMPERATURE = vgg_ckpt.get("temperature", 1.0)
    saved_val_acc = vgg_ckpt.get("val_accuracy", None)
    if "class_names" in vgg_ckpt:
        assert vgg_ckpt["class_names"] == CLASS_NAMES, "Class order in the VGG16 file is different!"
else:
    vgg_weights = vgg_ckpt
    VGG_TEMPERATURE = 1.0
    saved_val_acc = None

vgg_model = models.vgg16(weights=None)
vgg_model.classifier[6] = nn.Linear(4096, len(CLASS_NAMES))
vgg_model.load_state_dict(vgg_weights)
vgg_model = vgg_model.to(device)
vgg_model.eval()

# quick check on validation data
vgg_val_logits, vgg_val_labels = get_logits(vgg_model, val_loader)
vgg_val_acc = (vgg_val_logits.argmax(1) == vgg_val_labels).float().mean().item()
print("VGG16 validation accuracy now       :", round(vgg_val_acc, 4))
if saved_val_acc is not None:
    print("VGG16 validation accuracy when saved:", round(saved_val_acc, 4))
print("VGG16 temperature                   :", round(VGG_TEMPERATURE, 3))

If the two accuracies match (or are almost the same), the validation split is identical and the comparison with ResNet18 is fair.

In [ ]:
resnet_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet_model.fc = nn.Linear(512, len(CLASS_NAMES))       # new last layer with 4 outputs
resnet_model = resnet_model.to(device)

vgg_params = sum(p.numel() for p in vgg_model.parameters())
resnet_params = sum(p.numel() for p in resnet_model.parameters())
print("VGG16 parameters    :", round(vgg_params / 1e6, 1), "million")
print("ResNet18 parameters :", round(resnet_params / 1e6, 1), "million")

In [ ]:
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def run_epoch(model, loader, optimizer=None, freeze_bn=False):
    # one pass over the data. If an optimizer is given we train, otherwise we only evaluate
    training = optimizer is not None
    model.train(training)
    if training and freeze_bn:
        for m in model.modules():                 # keep BatchNorm statistics fixed while frozen
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

history = []
best = {"acc": 0.0, "loss": 1e9, "state": None}

def train_phase(phase_name, epochs, lr, freeze_backbone, early_stop):
    global best
    # freeze or unfreeze the network (the last layer 'fc' is always trained)
    for p in resnet_model.parameters():
        p.requires_grad = not freeze_backbone
    for p in resnet_model.fc.parameters():
        p.requires_grad = True

    trainable = [p for p in resnet_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1)

    bad_epochs = 0
    for epoch in range(1, epochs + 1):
        start = time.time()
        tr_loss, tr_acc = run_epoch(resnet_model, train_loader, optimizer, freeze_bn=freeze_backbone)
        va_loss, va_acc = run_epoch(resnet_model, val_loader)
        scheduler.step(va_loss)
        history.append({"train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc})

        # keep the best model (highest validation accuracy, lower loss if tied)
        is_best = (va_acc > best["acc"]) or (va_acc == best["acc"] and va_loss < best["loss"])
        if is_best:
            best = {"acc": va_acc, "loss": va_loss,
                    "state": {k: v.detach().cpu().clone() for k, v in resnet_model.state_dict().items()}}
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(phase_name, "| epoch", epoch, "/", epochs,
              "| train loss", round(tr_loss, 4), "acc", round(tr_acc, 4),
              "| val loss", round(va_loss, 4), "acc", round(va_acc, 4),
              "|", round(time.time() - start), "s", "<- best" if is_best else "")
        if early_stop and bad_epochs >= PATIENCE:
            print("Early stopping")
            break

train_start = time.time()
train_phase("Phase 1 (last layer)", FREEZE_EPOCHS, LR_HEAD, freeze_backbone=True, early_stop=False)
train_phase("Phase 2 (all layers)", FINETUNE_EPOCHS, LR_FINETUNE, freeze_backbone=False, early_stop=True)
resnet_train_minutes = (time.time() - train_start) / 60

resnet_model.load_state_dict(best["state"])      # go back to the best epoch
print("\nBest validation accuracy:", round(best["acc"], 4))
print("ResNet18 training time  :", round(resnet_train_minutes, 1), "minutes")

In [ ]:
# learning curves
epochs = range(1, len(history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="validation")
axes[0].set_title("ResNet18 loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs, [h["train_acc"] for h in history], label="train")
axes[1].plot(epochs, [h["val_acc"] for h in history], label="validation")
axes[1].set_title("ResNet18 accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "resnet18_training_curves.png", dpi=150)
plt.show()

## Step 8: Calibrate the confidence of ResNet18 (temperature scaling)
Neural networks are often **over-confident**. We divide the raw scores by one number `T` before softmax. We try many values of `T` on the **validation** data and keep the best one. This makes the confidence closer to the real accuracy, which matters because our web app sends low-confidence cases to a doctor.

VGG16 already has its own `T` saved in `vgg16.pth`.

In [ ]:
resnet_val_logits, resnet_val_labels = get_logits(resnet_model, val_loader)
RESNET_TEMPERATURE = find_temperature(resnet_val_logits, resnet_val_labels)

val_y = resnet_val_labels.numpy()
ece_before = calibration_error(torch.softmax(resnet_val_logits, 1).numpy(), val_y)
ece_after = calibration_error(torch.softmax(resnet_val_logits / RESNET_TEMPERATURE, 1).numpy(), val_y)
print("ResNet18 temperature T:", round(RESNET_TEMPERATURE, 2))
print("Calibration error on validation data:", round(ece_before, 4), "->", round(ece_after, 4))

# optional: save the ResNet18 weights (off by default, this notebook is separate from the project)
if SAVE_RESNET_WEIGHTS:
    torch.save(resnet_model.state_dict(), OUT_DIR / "resnet18_ipcv.pth")
    print("Saved ResNet18 weights in", OUT_DIR)

In [ ]:
report_path = OUT_DIR / "comparison_report.json"
if report_path.exists():
    print("Note: an old report exists and will be replaced. Do not change settings because of test results.")

test_loader = DataLoader(MRIDataset(test_items, eval_tf), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

def measure_speed(model):
    # average time to predict ONE image (after a short warm-up)
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
        times = []
        for _ in range(50):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            model(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times))

def evaluate_model(model, temperature):
    logits, labels = get_logits(model, test_loader)
    y_true = labels.numpy()
    probs_raw = torch.softmax(logits, 1).numpy()
    probs_cal = torch.softmax(logits / temperature, 1).numpy()
    y_pred = probs_cal.argmax(1)

    rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": rep["macro avg"]["precision"],
        "macro_recall": rep["macro avg"]["recall"],
        "macro_f1": rep["macro avg"]["f1-score"],
        "roc_auc": roc_auc_score(y_true, probs_cal, multi_class="ovr", average="macro"),
        "ece_before": calibration_error(probs_raw, y_true),
        "ece_after": calibration_error(probs_cal, y_true),
        "temperature": temperature,
        "precision_per_class": {c: rep[c]["precision"] for c in CLASS_NAMES},
        "recall_per_class": {c: rep[c]["recall"] for c in CLASS_NAMES},
        "f1_per_class": {c: rep[c]["f1-score"] for c in CLASS_NAMES},
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "parameters": sum(p.numel() for p in model.parameters()),
        "file_size_mb": sum(p.numel() for p in model.parameters()) * 4 / 1e6,     # 4 bytes per weight
        "inference_ms": measure_speed(model),
        "y_true": y_true, "probs_raw": probs_raw, "probs_cal": probs_cal,     # kept for the diagrams
    }
    return result

vgg_result = evaluate_model(vgg_model, VGG_TEMPERATURE)
resnet_result = evaluate_model(resnet_model, RESNET_TEMPERATURE)
resnet_result["training_minutes"] = resnet_train_minutes

# save everything (numpy arrays converted to lists)
to_save = {"vgg16": dict(vgg_result), "resnet18": dict(resnet_result)}
for name in to_save:
    to_save[name]["confusion_matrix"] = to_save[name]["confusion_matrix"].tolist()
    for key in ["y_true", "probs_raw", "probs_cal"]:      # big arrays are not saved in the JSON
        del to_save[name][key]
with open(report_path, "w") as f:
    json.dump(to_save, f, indent=2, default=float)
print("Saved:", report_path)

In [ ]:
# comparison table
rows = [
    ("Test accuracy",        "accuracy",        4),
    ("Macro precision",      "macro_precision", 4),
    ("Macro recall",         "macro_recall",    4),
    ("Macro F1-score",       "macro_f1",        4),
    ("ROC-AUC (macro)",      "roc_auc",         4),
    ("Calibration error before T", "ece_before", 4),
    ("Calibration error after T",  "ece_after",  4),
    ("Parameters (millions)", "parameters",     1),
    ("Model size (MB)",      "file_size_mb",    1),
    ("Time per image (ms)",  "inference_ms",    2),
]
print(f"{'Metric':<30}{'VGG16':>12}{'ResNet18':>12}")
print("-" * 54)
for title, key, digits in rows:
    a, b = vgg_result[key], resnet_result[key]
    if key == "parameters":
        a, b = a / 1e6, b / 1e6
    print(f"{title:<30}{round(a, digits):>12}{round(b, digits):>12}")

print("\nRecall per class (how many real cases of each class were found)")
print(f"{'Class':<30}{'VGG16':>12}{'ResNet18':>12}")
print("-" * 54)
for c in CLASS_NAMES:
    print(f"{c:<30}{round(vgg_result['recall_per_class'][c], 4):>12}{round(resnet_result['recall_per_class'][c], 4):>12}")

In [ ]:
# bar chart of the main scores
names = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
keys = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "roc_auc"]
vgg_scores = [vgg_result[k] for k in keys]
res_scores = [resnet_result[k] for k in keys]

x = np.arange(len(names))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, vgg_scores, width=0.4, label="VGG16")
plt.bar(x + 0.2, res_scores, width=0.4, label="ResNet18")
plt.xticks(x, names)
plt.ylim(min(vgg_scores + res_scores) - 0.1, 1.0)
plt.title("VGG16 vs ResNet18 on the test set")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "comparison_bar_chart.png", dpi=150)
plt.show()

# confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, result, title in zip(axes, [vgg_result, resnet_result], ["VGG16", "ResNet18"]):
    cm = result["confusion_matrix"]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title + " confusion matrix")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig(OUT_DIR / "comparison_confusion_matrices.png", dpi=150)
plt.show()

**How to read the diagrams below**
* **Per-class bars:** which classes each model finds well or badly.
* **ROC curves:** the closer a curve is to the top-left corner, the better the model separates that class from the others.
* **Reliability diagram:** the closer the line is to the diagonal, the more honest the confidence values are.
* **Cost chart:** size and speed of the two models.

In [ ]:
# per-class precision, recall and F1 for both models
metric_list = [("Precision", "precision_per_class"), ("Recall", "recall_per_class"), ("F1-score", "f1_per_class")]
x = np.arange(len(CLASS_NAMES))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (title, key) in zip(axes, metric_list):
    ax.bar(x - 0.2, [vgg_result[key][c] for c in CLASS_NAMES], width=0.4, label="VGG16")
    ax.bar(x + 0.2, [resnet_result[key][c] for c in CLASS_NAMES], width=0.4, label="ResNet18")
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=20)
    ax.set_title(title + " per class")
    ax.set_ylim(0.5, 1.02)
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "per_class_metrics.png", dpi=150)
plt.show()

In [ ]:
# ROC curves: each class against all other classes
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
for k, class_name in enumerate(CLASS_NAMES):
    for result, name in [(vgg_result, "VGG16"), (resnet_result, "ResNet18")]:
        is_class = (result["y_true"] == k).astype(int)
        fpr, tpr, _ = roc_curve(is_class, result["probs_cal"][:, k])
        axes[k].plot(fpr, tpr, label=name + " (AUC " + str(round(auc(fpr, tpr), 3)) + ")")
    axes[k].plot([0, 1], [0, 1], "k--", linewidth=0.8)
    axes[k].set_title(class_name + " vs rest")
    axes[k].set_xlabel("False positive rate")
    axes[k].legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("True positive rate")
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curves.png", dpi=150)
plt.show()

In [ ]:
# reliability diagram: confidence versus real accuracy
def reliability_points(probs, labels, n_bins=10):
    confidence = probs.max(1)
    correct = (probs.argmax(1) == labels)
    edges = np.linspace(0, 1, n_bins + 1)
    xs, ys = [], []
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.sum() > 0:
            xs.append(confidence[in_bin].mean())
            ys.append(correct[in_bin].mean())
    return xs, ys

plt.figure(figsize=(5.5, 5))
plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
for result, name in [(vgg_result, "VGG16"), (resnet_result, "ResNet18")]:
    xs, ys = reliability_points(result["probs_cal"], result["y_true"])
    plt.plot(xs, ys, "o-", label=name + " (error " + str(round(result["ece_after"], 3)) + ")")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability diagram (test set)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "reliability_diagram.png", dpi=150)
plt.show()

In [ ]:
# cost of the two models
cost_items = [("Parameters (millions)", "parameters", 1e6),
              ("Model size (MB)", "file_size_mb", 1),
              ("Time per image (ms)", "inference_ms", 1)]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (title, key, divide) in zip(axes, cost_items):
    values = [vgg_result[key] / divide, resnet_result[key] / divide]
    ax.bar(["VGG16", "ResNet18"], values, color=["C0", "C1"])
    for i, v in enumerate(values):
        ax.text(i, v, str(round(v, 1)), ha="center", va="bottom")
    ax.set_title(title)
plt.tight_layout()
plt.savefig(OUT_DIR / "cost_comparison.png", dpi=150)
plt.show()

In [ ]:
# short automatic summary of the comparison
better = "ResNet18" if resnet_result["accuracy"] > vgg_result["accuracy"] else "VGG16"
print("Higher test accuracy :", better,
      "(VGG16", round(vgg_result["accuracy"] * 100, 2), "% | ResNet18", round(resnet_result["accuracy"] * 100, 2), "%)")
print("Size                 : VGG16 has", round(vgg_result["parameters"] / resnet_result["parameters"], 1), "times more parameters than ResNet18")
print("Model size           : VGG16", round(vgg_result["file_size_mb"]), "MB | ResNet18", round(resnet_result["file_size_mb"]), "MB")
faster = "ResNet18" if resnet_result["inference_ms"] < vgg_result["inference_ms"] else "VGG16"
print("Faster prediction    :", faster,
      "(VGG16", round(vgg_result["inference_ms"], 1), "ms | ResNet18", round(resnet_result["inference_ms"], 1), "ms)")
print("\nNow compare the confusion matrices: which model misses fewer tumours (glioma or meningioma predicted as notumor)?")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# in-place ReLU can break the gradient hooks, so switch it off (predictions stay the same)
for model in [vgg_model, resnet_model]:
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            m.inplace = False

vgg_layer = vgg_model.features[28]        # last convolution layer of VGG16
resnet_layer = resnet_model.layer4[-1]    # last block of ResNet18

def localise(model, layer, temperature, path):
    # runs the model, then the image processing steps. Returns every step in a dictionary
    pil_img = Image.open(path).convert("RGB")
    input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)
    rgb = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0

    # prediction
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(input_tensor).float().cpu() / temperature, dim=1)[0]
    label = CLASS_NAMES[int(probs.argmax())]
    confidence = float(probs.max())

    # step 1: Grad-CAM heatmap -> 8-bit image
    with GradCAM(model=model, target_layers=[layer]) as cam:
        heatmap = cam(input_tensor=input_tensor, targets=None)[0]
    heat_u8 = (heatmap * 255).astype(np.uint8)
    overlay = show_cam_on_image(rgb, heatmap, use_rgb=True)

    # step 2: Otsu threshold
    _, mask = cv2.threshold(heat_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # steps 3 and 4: opening, then closing
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

    # steps 5 and 6: contours and bounding box of the biggest region
    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = (rgb * 255).astype(np.uint8).copy()
    box = None
    if len(contours) > 0:
        biggest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(biggest)
        cv2.drawContours(result, [biggest], -1, (0, 255, 0), 2)           # green outline
        cv2.rectangle(result, (x, y), (x + w, y + h), (255, 0, 0), 2)      # red box
        box = (x, y, w, h)

    return {"rgb": rgb, "overlay": overlay, "mask": mask, "mask_clean": mask_clean,
            "result": result, "label": label, "confidence": confidence, "box": box,
            "area_percent": 100.0 * float((mask_clean > 0).mean())}

In [ ]:
# step-by-step example with the more accurate model
if resnet_result["accuracy"] > vgg_result["accuracy"]:
    demo_name, demo_model, demo_layer, demo_T = "ResNet18", resnet_model, resnet_layer, RESNET_TEMPERATURE
else:
    demo_name, demo_model, demo_layer, demo_T = "VGG16", vgg_model, vgg_layer, VGG_TEMPERATURE

demo_class = "glioma"
for path in [p for p, c in test_items if c == demo_class][:30]:     # first image the model gets right
    demo = localise(demo_model, demo_layer, demo_T, path)
    if demo["label"] == demo_class:
        break

panels = [(demo["rgb"], "1. Original"), (demo["overlay"], "2. Grad-CAM heatmap"),
          (demo["mask"], "3. Otsu threshold"), (demo["mask_clean"], "4. Opening + closing"),
          (demo["result"], "5. Contour + box")]
fig, axes = plt.subplots(1, 5, figsize=(17, 3.9))
for ax, (image, title) in zip(axes, panels):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle(demo_name + " predicts " + demo["label"] + " (" + str(round(demo["confidence"] * 100)) + "%)", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "localisation_steps.png", dpi=150, bbox_inches="tight")
plt.show()

print("Bounding box (x, y, width, height):", demo["box"])
print("Highlighted area:", round(demo["area_percent"], 1), "% of the image")

In [ ]:
# both models on the same images, one row per tumour class
tumour_classes = ["glioma", "meningioma", "pituitary"]
fig, axes = plt.subplots(3, 3, figsize=(11, 11))

for row, class_name in enumerate(tumour_classes):
    for path in [p for p, c in test_items if c == class_name][:30]:
        res_v = localise(vgg_model, vgg_layer, VGG_TEMPERATURE, path)
        res_r = localise(resnet_model, resnet_layer, RESNET_TEMPERATURE, path)
        if res_v["label"] == class_name and res_r["label"] == class_name:
            break                         # stop at the first image both models get right

    shown = [(res_v["rgb"], "Original (" + class_name + ")"),
             (res_v["result"], "VGG16: " + res_v["label"] + " " + str(round(res_v["confidence"] * 100)) + "%"),
             (res_r["result"], "ResNet18: " + res_r["label"] + " " + str(round(res_r["confidence"] * 100)) + "%")]
    for col, (image, title) in enumerate(shown):
        axes[row, col].imshow(image, cmap="gray")
        axes[row, col].set_title(title)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "localisation_comparison.png", dpi=150)
plt.show()